In [1]:
# baseline random forest classifier, metrics, and wrapper

In [1]:
# imports/load data
import os
import pickle
import sqlite3
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
from database import FairDB, calc_batch_fairness, check_violations

In [2]:
"""
VARIABLE KEYS: (for specific demo)
protected = protected_attribute binary (GENDER)
privileged = MALE
unprivileged = FEMALE
unpriv_mask = f_mask (FEMALE)
priv_mask = m_mask (MALE)

"""

'\nVARIABLE KEYS: (for specific demo)\nprotected = protected_attribute binary (GENDER)\nprivileged = MALE\nunprivileged = FEMALE\nunpriv_mask = f_mask (FEMALE)\npriv_mask = m_mask (MALE)\n\n'

In [4]:
# generic config class
class FairnessConfig:
    """ 
    Configuration class for relevant fairness-monitoring tasks. User must specify
    datapath, target column(s), and protected attribute(s)

    args:
        data_path: path for specific dataset (.csv file)
        target: name of target column (ex: 'income')
        protected_attr: name of protected attribute (ex: 'sex')
        """
    def __init__(self, data_path, target, protected_attr, desired=None, privileged=None):
        self.data_path = data_path
        # load data from path
        self.df = pd.read_csv(data_path, na_values='?', header=None)

        self.target = target
        self.protected_attr = protected_attr
        
        if isinstance(self.df.columns[0], int):
            self.df = pd.read_csv(data_path, na_values='?', header=0)

        # auto-detect desired & privileged labels (binary 0/1, auto-selects second val alphabetically)
        if desired is None:
            vals = sorted(self.df[target].dropna().unique())
            self.desired = vals[-1]  # ex: 0/1 for >/< 50k
            print(f"auto-detected desired outcome: {self.desired}")
        else:
            self.desired = desired

        if privileged is None:
            vals = sorted(self.df[protected_attr].dropna().unique())
            self.privileged = vals[-1]  # ex: 'male', 'female'
            print(f"auto-detected privileged group: {self.privileged}")
        else:
            self.privileged = privileged

        # auto-detect feature cols, numeric vs categorical
        self.feature_cols = [c for c in self.df.columns if c not in [self.target, self.protected_attr]]
        self.numeric_cols = self.df[self.feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
        self.categorical_cols = self.df[self.feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

        print(f"\nauto-detected {len(self.feature_cols)} feature cols, {len(self.numeric_cols)} numeric cols, and {len(self.categorical_cols)} categorical cols")

    def to_dict(self):
        """ export to dictionary for .json """
        return {
            'data_path': self.data_path,
            'target': self.target,
            'protected_attr': self.protected_attr,
            'desired_label': self.desired,
            'privileged_label': self.privileged,
            'feature_cols': self.feature_cols,
            'numeric_cols': self.numeric_cols,
            'categorical_cols': self.categorical_cols }


In [5]:
with open('processed_data.pkl', 'rb') as f: data = pickle.load(f)

# train/test/fair
X_train = data['X_train']
y_train = data['y_train']
protected_train = data['protected_train']  # binary og
protected_raw_train = data['protected_raw_train']  # non-binary og

X_test = data['X_test']
y_test = data['y_test']
protected_test = data['protected_test']
protected_raw_test = data['protected_raw_test']

X_fair = data['X_fair']
y_fair = data['y_fair']
protected_fair = data['protected_fair']
protected_raw_fair = data['protected_raw_fair']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Fairness set: {X_fair.shape}")

Train set: (19536, 99)
Test set: (6512, 99)
Fairness set: (4316, 99)


In [6]:
# baseline random forest classifier initialiized w balanced weights
base_model = RandomForestClassifier(
    n_estimators = 200, # tree num
    max_depth = 15, # overfitting
    min_samples_split = 10,
    min_samples_leaf = 5,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs = -1,
    verbose = 1)

# train model
base_model.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.4s finished


,n_estimators,200
,criterion,'gini'
,max_depth,15
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
# metrics
def evaluate(model, X, y, name):
    """
    evaluate base performance metrics
    """
    
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    print(f"{name} set performance")

    # meterics
    print(f"\nAccuracy: {accuracy_score(y, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y, y_prob):.4f}")
    print(f"\nClassification report:")
    print(classification_report(y, y_pred, target_names= ['<=50K', '>50K']))

    # confusion matrix
    cm = confusion_matrix(y,  y_pred)
    print(f"\nConfusion matrix:")
    print(f"               Predicted <=50K      Predicted >50K")
    print(f"Actual <=50K   {cm[0,0]:6d}        {cm[0,1]:6d}")
    print(f"Actual >50K    {cm[1,0]:6d}        {cm[1,1]:6d}")

    return {
    'accuracy' : float(accuracy_score(y, y_pred)),
    'roc_auc' : float(roc_auc_score(y, y_prob)),
    'confusion_matrix' : cm.tolist() }

# evaluate on all sets
train_perf = evaluate(base_model, X_train, y_train,"Train")
test_perf = evaluate(base_model, X_test, y_test, "Test")
fair_perf = evaluate(base_model, X_fair, y_fair, "Fair")

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.


Train set performance

Accuracy: 0.8149
ROC-AUC: 0.9278

Classification report:
              precision    recall  f1-score   support

       <=50K       0.96      0.79      0.87     14831
        >50K       0.57      0.90      0.70      4705

    accuracy                           0.81     19536
   macro avg       0.77      0.85      0.78     19536
weighted avg       0.87      0.81      0.83     19536


Confusion matrix:
               Predicted <=50K      Predicted >50K
Actual <=50K    11666          3165
Actual >50K       451          4254
Test set performance

Accuracy: 0.7970
ROC-AUC: 0.9084

Classification report:
              precision    recall  f1-score   support

       <=50K       0.95      0.78      0.85      4944
        >50K       0.55      0.87      0.67      1568

    accuracy                           0.80      6512
   macro avg       0.75      0.82      0.76      6512
weighted avg       0.85      0.80      0.81      6512


Confusion matrix:
               Predicted <

[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


Fair set performance

Accuracy: 0.8276
ROC-AUC: 0.9203

Classification report:
              precision    recall  f1-score   support

       <=50K       0.96      0.82      0.88      3396
        >50K       0.56      0.87      0.68       920

    accuracy                           0.83      4316
   macro avg       0.76      0.84      0.78      4316
weighted avg       0.87      0.83      0.84      4316


Confusion matrix:
               Predicted <=50K      Predicted >50K
Actual <=50K     2775           621
Actual >50K       123           797


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [8]:
def calc_fairness(y_true, y_pred, protected, name = ""):
    """
    calculate fairness for all sets
    
    negative - disadvantage for unprivilegeds
    pos - advantage for unprivilegeds

    """
    
    # separate by protected_attr
    unpriv_mask = (protected == 0)
    priv_mask = (protected == 1)

    # pred/true labels
    unpriv_pred = y_pred[unpriv_mask] # unprivileged y (fy)
    unpriv_true = y_true[unpriv_mask]
    priv_pred = y_pred[priv_mask] # privileged y (my)
    priv_true = y_true[priv_mask]

    metrics = {}

    # stat parity/sel rate difference
    unpriv_pos_rate = unpriv_pred.mean()
    priv_pos_rate = priv_pred.mean()
    stat_parity = unpriv_pos_rate - priv_pos_rate

    metrics['statistical_parity_difference'] = float(stat_parity)
    metrics['unprivileged_sel_rate'] = float(unpriv_pos_rate)
    metrics['privileged_sel_rate'] = float(priv_pos_rate)

    # equal opportunity (true pos rate difference)
    unpriv_pos = unpriv_true.sum()
    priv_pos = priv_true.sum()

    if unpriv_pos > 0 and priv_pos > 0: # determining true pos rate (tpr)
        unpriv_tpr = ((unpriv_pred == 1) & (unpriv_true == 1)).sum() / unpriv_pos
        priv_tpr = ((priv_pred == 1) & (priv_true == 1)).sum() / priv_pos
        eq_op = unpriv_tpr - priv_tpr # equal opportunity 

        metrics['equal_opportunity_difference'] = float(eq_op)
        metrics['unprivileged_tpr'] = float(unpriv_tpr)
        metrics['privileged_tpr'] = float(priv_tpr)

    # predictive parity/precision difference
    unpriv_pred_pos = unpriv_pred.sum() # predicted positive
    priv_pred_pos = priv_pred.sum()

    if unpriv_pred_pos > 0 and priv_pred_pos > 0:
        unpriv_precision = ((unpriv_pred == 1) & (unpriv_true == 1)).sum() / unpriv_pred_pos
        priv_precision = ((priv_pred == 1) & (priv_true == 1)).sum() / priv_pred_pos
        pred_parity = unpriv_precision - priv_precision

        metrics['predictive_parity_difference'] = float(pred_parity)
        metrics['unprivileged_precision'] = float(unpriv_precision)
        metrics['privileged_precision'] = float(priv_precision)

    # accuracy equality
    unpriv_acc = (unpriv_pred == unpriv_true).mean()
    priv_acc = (priv_pred == priv_true).mean()
    acc_diff = unpriv_acc - priv_acc

    metrics['accuracy_difference'] = float(acc_diff)
    metrics['unprivileged_accuracy'] = float(unpriv_acc)
    metrics['privileged_accuracy'] = float(priv_acc)

    return metrics

# calculate for all sets
train_fairness = calc_fairness(y_train, base_model.predict(X_train), protected_train)
test_fairness = calc_fairness(y_test, base_model.predict(X_test), protected_test)
fair_fairness = calc_fairness(y_fair, base_model.predict(X_fair), protected_fair)

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [9]:
# print results
for name, fairness in [('Train', train_fairness), ('Test', test_fairness), ('Fair', fair_fairness)]:
    print(f"\n{'-' * 50}")
  #  print(f"\n{set_name} Set Fairness Metrics")
    
    print(f"\nStatistical Parity/Selection Rate Difference")
    print(f"unprivileged sel rate: {fairness['unprivileged_sel_rate']:.4f}")
    print(f"privileged sel rate: {fairness['privileged_sel_rate']:.4f}")
    print(f"Difference (f - m): {fairness['statistical_parity_difference']:.4f}")
    print(f" -- {'Fair' if abs(fairness['statistical_parity_difference']) < 0.05 else 'Unfair'}")

    if 'equal_opportunity_difference' in fairness:
        print(f"\nEqual Opportunity/TPR Difference")
        print(f"unprivileged tpr: {fairness['unprivileged_tpr']:.4f}")
        print(f"privileged tpr: {fairness['privileged_tpr']:.4f}")
        print(f"Difference (f - m): {fairness['equal_opportunity_difference']:.4f}")
        print(f" -- {'Fair' if abs(fairness['equal_opportunity_difference']) < 0.05 else 'Unfair'}")

    if 'predictive_parity_difference' in fairness:
        print(f"\nPredictive Parity/Precision Difference")
        print(f"unprivileged precision: {fairness['unprivileged_precision']:.4f}")
        print(f"privileged precision: {fairness['privileged_precision']:.4f}")
        print(f"Difference (f - m): {fairness['predictive_parity_difference']:.4f}")
        print(f" -- {'Fair' if abs(fairness['predictive_parity_difference']) < 0.05 else 'Unfair'}")

    print(f"\nAccuracy Equality")
    print(f"unprivileged accuracy: {fairness['unprivileged_accuracy']:.4f}")
    print(f"privileged accuracy: {fairness['privileged_accuracy']:.4f}")
    print(f"Difference (f - m): {fairness['accuracy_difference']:.4f}")
    print(f" -- {'Fair' if abs(fairness['accuracy_difference']) < 0.05 else 'Unfair'}")



--------------------------------------------------

Statistical Parity/Selection Rate Difference
unprivileged sel rate: 0.1451
privileged sel rate: 0.4966
Difference (f - m): -0.3514
 -- Unfair

Equal Opportunity/TPR Difference
unprivileged tpr: 0.8245
privileged tpr: 0.9181
Difference (f - m): -0.0935
 -- Unfair

Predictive Parity/Precision Difference
unprivileged precision: 0.6136
privileged precision: 0.5675
Difference (f - m): 0.0460
 -- Fair

Accuracy Equality
unprivileged accuracy: 0.9250
privileged accuracy: 0.7601
Difference (f - m): 0.1649
 -- Unfair

--------------------------------------------------

Statistical Parity/Selection Rate Difference
unprivileged sel rate: 0.1499
privileged sel rate: 0.4901
Difference (f - m): -0.3402
 -- Unfair

Equal Opportunity/TPR Difference
unprivileged tpr: 0.7339
privileged tpr: 0.8891
Difference (f - m): -0.1552
 -- Unfair

Predictive Parity/Precision Difference
unprivileged precision: 0.5377
privileged precision: 0.5516
Difference (f - m

In [10]:
class FairnessModel:
    """
    baseline wrapper func for model integration
    """
    
    def __init__(self, model, name="baseline_model", use_db=False, db_path='fair.db'):
        """ 
        initalize with trained model
        """
        
        self.model = model 
        self.name = name
        self.use_db = use_db
        self.prediction_log = []

        # create database/use memeory
        if use_db:
            self.db = FairDB(db_path)
        else:
            self.prediction_log = []
        self.batch_id = 1 # to track batch 

    def predict(self, X, protected_attr = None, metadata = None, true_labels=None, batch_id=None, calc_metrics = False):
        """
        automatically log predictions made
        """
        
        predictions = self.model.predict(X)
        # determine batch_id
        if batch_id is None:
            batch_id = self.batch_id
            self.batch_id += 1

        if self.use_db:
            self.db.log_batch(predictions, protected_attr, self.name, true_labels, batch_id)

            # calc  metrics if needed
            if calc_metrics and protected_attr is not None:
                preds_df = self.db.get_batch_preds(batch_id)
                metrics = calc_batch_fairness(preds_df)

                if metrics:
                    self.db.log_batch_metrics(batch_id, self.name, metrics, len(predictions))
                    # check violations
                    viol = check_violations(metrics)
                    for v in viol:
                        self.db.log_alert(batch_id, v['metric_name'], v['value'], v['threshold'], v['severity'], v['label'])
        else:
            # log memory
            for i in range(len(predictions)):
                log_entry = {
                    'prediction' : int(predictions[i]),
                    'timestamp' : pd.Timestamp.now(),
                    'model_name' : self.name }
        
                if protected_attr is not None:
                    log_entry['protected_attr'] = int(
                        protected_attr.iloc[i] if hasattr(protected_attr, 'iloc') 
                        else protected_attr [i] )
                    
                if metadata is not None: log_entry.update(metadata)
                self.prediction_log.append(log_entry)
        
        return predictions

    def probability(self, X, protected_attr = None):
        """
        run probability for predictions
        """
        
        return self.model.predict(X)

    def get_log(self):
        """
         get logged pred as dataFrame
         """
        
        if self.use_db:
            return self.db.get_preds(limit=10000)
        else:
            return pd.DataFrame(self.prediction_log)

    def get_batch_metrics(self):
        """
        run and return metrics for batch
        """
        
        if self.use_db:
            return self.db.get_batch_metrics()
        else:
            print("db not enabled")
            return pd.DataFrame()

    def get_alerts(self):
        if self.use_db:
            return self.db.get_alerts()
        else:
            print("db not enabled")
            return pd.DataFrame()
            
    def clear_log(self):
        self.prediction_log = []

    def close(self):
        if self.use_db:
            self.db.close()

In [11]:
# test wrapper on testing  set
wrapped_model = FairnessModel(base_model, name = "baseline_rf")
wrap_test = wrapped_model.predict(X_test, protected_attr = protected_test)
log_df = wrapped_model.get_log()

print(f"Logged {len(log_df)} predictions from test set")
print(log_df.head())

Logged 6512 predictions from test set
   prediction                  timestamp   model_name  protected_attr
0           1 2026-04-07 20:46:34.553511  baseline_rf               0
1           0 2026-04-07 20:46:34.553631  baseline_rf               1
2           0 2026-04-07 20:46:34.553644  baseline_rf               1
3           1 2026-04-07 20:46:34.553652  baseline_rf               1
4           1 2026-04-07 20:46:34.553658  baseline_rf               1


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [12]:
# verify behavior matches original model
og_pred = base_model.predict(X_test)
wrap_pred = wrap_test
wrap_fair = calc_fairness(y_test, wrap_pred, protected_test)

print(f"\nPredictions match og model: {np.array_equal(wrap_pred, og_pred)}")
print(f"Fairness metrics comparison:")
print(f"Og statistical parity: {test_fairness['statistical_parity_difference']:.4f}")
print(f"Wrap statistical parity: {wrap_fair['statistical_parity_difference']:.4f}")
print(f"Match: {abs(test_fairness['statistical_parity_difference'] - wrap_fair['statistical_parity_difference'] < 0.0001)}")


Predictions match og model: True
Fairness metrics comparison:
Og statistical parity: -0.3402
Wrap statistical parity: -0.3402
Match: 1


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [13]:
# save metrics
with open('baseline_model.pkl', 'wb') as f:
    pickle.dump(base_model, f)

# comprehensive baseline metrics (for drift detection)
baseline_metrics = {
    'model_info': {
        'model_type': 'RandomForestClassifier',
        'model_name': 'baseline_rf',
        'trained_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'training_samples': int(X_train.shape[0]),
        'n_features': int(X_train.shape[1]),
        'hyperparameters': {
            'n_estimators': 200,
            'max_depth': 15,
            'class_weight': 'balanced'
        }},
    'performance': {
        'train': train_perf,
        'test': test_perf,
        'fairness_set': fair_perf
    },
    'fairness_baseline': {
        'train': train_fairness,
        'test': test_fairness,
        'fairness_set': fair_fairness,  # drift detect
        'threshold': 0.05,
        'description': 'Baseline fairness from protected-balanced fairness set'
    }
}

with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print("\nBaseline Fairness Summary (Fairness Set):")
print(f"  Statistical Parity:   {fair_fairness['statistical_parity_difference']:.4f} (UNFAIR)")
print(f"  Equal Opportunity:    {fair_fairness['equal_opportunity_difference']:.4f} (UNFAIR)")
print(f"  Predictive Parity:    {fair_fairness['predictive_parity_difference']:.4f} (FAIR)")
print(f"  Accuracy Difference:  {fair_fairness['accuracy_difference']:.4f} (UNFAIR)")


Baseline Fairness Summary (Fairness Set):
  Statistical Parity:   -0.3401 (UNFAIR)
  Equal Opportunity:    -0.0681 (UNFAIR)
  Predictive Parity:    0.0300 (FAIR)
  Accuracy Difference:  0.1715 (UNFAIR)


In [14]:
# test model/db wrapper
try: 
    wrapped_model.close()
except:
    pass
    
# del old db
db_name = f'test_{pd.Timestamp.now().strftime("%H%M%S")}.db'

wrapped_model = FairnessModel(base_model, name="baseline_rf", use_db=True, db_path=db_name)

# test: log pred in batches
batch_size = 100

for batch_num in range(5):
    start = batch_num * batch_size
    end = start + batch_size
    
    X_batch = X_test.iloc[start:end]
    protected_batch = protected_test.iloc[start:end]
    y_batch = y_test.iloc[start:end]
    
    preds = wrapped_model.predict(
        X_batch,
        protected_attr=protected_batch,
        true_labels=y_batch,
        batch_id=batch_num + 1,
        calc_metrics=True
    )
    
    print(f"  batch {batch_num + 1}: {len(preds)} preds")

# test: view recent preds
recent = wrapped_model.get_log()
print(recent.head(10))

# test: view batch metrics
batch_metrics = wrapped_model.get_batch_metrics()
print(batch_metrics[['batch_id', 'stat_diff', 'eq_opp_diff', 'acc_diff']])

# test: view alerts
alerts = wrapped_model.get_alerts()
if len(alerts) > 0:
    print(alerts[['batch_id', 'metric_name', 'metric_val', 'severity']])
else:
    print("  no alerts")

# test: specific batch
batch_1 = wrapped_model.db.get_batch_preds(1)
print(f"  total: {len(batch_1)}")
print(f"  unprivileged: {(batch_1['protected_attr'] == 0).sum()}")
print(f"  privileged: {(batch_1['protected_attr'] == 1).sum()}")
print(f"  positive preds: {(batch_1['prediction'] == 1).sum()}")

wrapped_model.close()

Database created
Logged 100 predictions
logged metrics (batch: 1)
ALERT: critical - statistical_parity_difference = -0.2590
ALERT: critical - equal_opportunity_difference = -0.4500
ALERT: critical - predictive_parity_difference = -0.2552
  batch 1: 100 preds
Logged 100 predictions
logged metrics (batch: 2)
ALERT: critical - statistical_parity_difference = -0.3264
ALERT: critical - equal_opportunity_difference = -0.1250
ALERT: critical - predictive_parity_difference = 0.2500
ALERT: critical - accuracy_difference = 0.1944
  batch 2: 100 preds
Logged 100 predictions
logged metrics (batch: 3)
ALERT: critical - statistical_parity_difference = -0.4150
ALERT: critical - equal_opportunity_difference = -0.6071
ALERT: critical - predictive_parity_difference = 0.4545
ALERT: critical - accuracy_difference = 0.1355
  batch 3: 100 preds
Logged 100 predictions
logged metrics (batch: 4)
ALERT: critical - statistical_parity_difference = -0.4066
ALERT: critical - equal_opportunity_difference = -0.4286


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Do

ALERT: warning - accuracy_difference = 0.0620
  batch 4: 100 preds
Logged 100 predictions
logged metrics (batch: 5)
ALERT: critical - statistical_parity_difference = -0.3905
ALERT: critical - equal_opportunity_difference = -0.1600
ALERT: critical - predictive_parity_difference = 0.1846
ALERT: critical - accuracy_difference = 0.1619
  batch 5: 100 preds
    id                   timestamp   model_name  prediction  true_label  \
0  401  2026-04-07 20:46:45.022296  baseline_rf           0           0   
1  402  2026-04-07 20:46:45.022296  baseline_rf           0           0   
2  403  2026-04-07 20:46:45.022296  baseline_rf           1           1   
3  404  2026-04-07 20:46:45.022296  baseline_rf           1           0   
4  405  2026-04-07 20:46:45.022296  baseline_rf           0           0   
5  406  2026-04-07 20:46:45.022296  baseline_rf           1           0   
6  407  2026-04-07 20:46:45.022296  baseline_rf           0           0   
7  408  2026-04-07 20:46:45.022296  baseline_

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [15]:
def inject_bias(X, protected, intensity=0.3):
    """
    bias injection- degrade features for unprivilegeds to simulate drift
    """
    
    # X as feature df
    X_biased = X.copy()
    unpriv_mask = (protected == 0)

    # degrade numeric columns
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    for col in numeric_cols:
        X_biased.loc[unpriv_mask, col] = X.loc[unpriv_mask, col] * (1 - intensity)

    print(f"degraded {len(numeric_cols)} numeric features by {intensity*100:.0f}%")

    return X_biased
 
def gradual_bias(X, protected, batch_num, max_batches=10):
    """
    simulate slow drift by gradually worstening bias over batches
    """
    
    intensity = min(batch_num / max_batches, 1.0) * 0.5 # max = 50%
    print(f"Batch {batch_num}: intensity {intensity:.2f}")
    return inject_bias(X, protected, intensity)

In [16]:
print("Checking protected_fair data:")
print(f"Total rows: {len(protected_fair)}")
print(f"unprivileged (0): {(protected_fair == 0).sum()}")
print(f"privileged (1): {(protected_fair == 1).sum()}")
print(f"\nFirst 100 rows:")
print(f"unprivileged: {(protected_fair.iloc[0:100] == 0).sum()}")
print(f"privileged: {(protected_fair.iloc[0:100] == 1).sum()}")
print(f"\nUnique values in protected_fair: {protected_fair.unique()}")

# close prev conn
try:
    wrapped.close()
except:
    pass

# del old db
db_name = f'test_{pd.Timestamp.now().strftime("%H%M%S")}.db'

# test bias injection
wrapped = FairnessModel(base_model, name="baseline_rf", use_db=True, db_path = db_name)

# batch 1 , no bias
X1 = X_fair.iloc[0:100].copy()
g1 = protected_fair.iloc[0:100]
y1 = y_fair.iloc[0:100]
pred1 = wrapped.predict(X1, protected_attr=g1, true_labels=y1, batch_id=1, calc_metrics=True)
m1 = calc_batch_fairness(wrapped.db.get_batch_preds(1))
print(f" stat parity: {m1['statistical_parity_difference']:.4f}")

# batches 2-5 , gradual bias
for batch in range(2, 6):
    start = batch * 100
    end = start + 100

    X_batch = X_fair.iloc[start:end].copy()
    g_batch = protected_fair.iloc[start:end]
    y_batch = y_fair.iloc[start:end]

    # inject bias
    X_biased = gradual_bias(X_batch, g_batch, batch, max_batches=10)
    preds = wrapped.predict(X_biased, protected_attr=g_batch, true_labels=y_batch, batch_id=batch, calc_metrics=True)
    metrics = calc_batch_fairness(wrapped.db.get_batch_preds(batch))
    print(f" batch {batch} stat parity: {metrics['statistical_parity_difference']:.4f}")

# all metrics & alerts 
all_metrics = wrapped.get_batch_metrics()
print(all_metrics[['batch_id', 'stat_diff', 'eq_opp_diff', 'acc_diff',]])
alerts = wrapped.get_alerts()
if len(alerts) > 0:
    print(alerts[['batch_id', 'metric_name', 'metric_val', 'severity']])
else:
    print("no alerts")

wrapped.close()

Checking protected_fair data:
Total rows: 4316
unprivileged (0): 2158
privileged (1): 2158

First 100 rows:
unprivileged: 52
privileged: 48

Unique values in protected_fair: [0 1]
Database created
Logged 100 predictions
logged metrics (batch: 1)
ALERT: critical - statistical_parity_difference = -0.3670
ALERT: critical - equal_opportunity_difference = 0.1875
ALERT: critical - accuracy_difference = 0.2356
 stat parity: -0.3670
Batch 2: intensity 0.10
degraded 99 numeric feaetures by 10%
Logged 100 predictions
logged metrics (batch: 2)
ALERT: critical - statistical_parity_difference = -0.2596
ALERT: warning - equal_opportunity_difference = 0.0625
ALERT: warning - predictive_parity_difference = -0.0833
ALERT: warning - accuracy_difference = 0.0737
 batch 2 stat parity: -0.2596
Batch 3: intensity 0.15
degraded 99 numeric feaetures by 15%


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Do

Logged 100 predictions
logged metrics (batch: 3)
ALERT: critical - statistical_parity_difference = -0.2806
ALERT: critical - equal_opportunity_difference = -0.1583
ALERT: critical - predictive_parity_difference = -0.2525
 batch 3 stat parity: -0.2806
Batch 4: intensity 0.20
degraded 99 numeric feaetures by 20%
Logged 100 predictions
logged metrics (batch: 4)
ALERT: critical - statistical_parity_difference = -0.5542
ALERT: critical - equal_opportunity_difference = -0.1974
ALERT: critical - predictive_parity_difference = 0.1694
ALERT: critical - accuracy_difference = 0.2465
 batch 4 stat parity: -0.5542
Batch 5: intensity 0.25
degraded 99 numeric feaetures by 25%
Logged 100 predictions
logged metrics (batch: 5)
ALERT: critical - statistical_parity_difference = -0.3141
ALERT: warning - equal_opportunity_difference = 0.0667
ALERT: critical - predictive_parity_difference = 0.2400
ALERT: critical - accuracy_difference = 0.2057
 batch 5 stat parity: -0.3141
   batch_id  stat_diff  eq_opp_diff

[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished


In [17]:
# run some batches to generate data
wrapped = FairnessModel(base_model, name="export", use_db=True, db_path='website_data.db')

print("Generating demo data for website...")
for batch in range(1, 11):
    start = batch * 100
    end = start + 100
    
    X_batch = X_fair.iloc[start:end].copy()
    y_batch = y_fair.iloc[start:end]
    g_batch = protected_fair.iloc[start:end]
    
    # inject bias after batch 5
    if batch > 5:
        X_batch = inject_bias(X_batch, g_batch, intensity=0.4)
    
    preds = wrapped.predict(X_batch, protected_attr=g_batch, 
                           true_labels=y_batch, batch_id=batch, calc_metrics=True)
    print(f"  batch {batch} complete")

# export to JSON
metrics_history = wrapped.get_batch_metrics()
alerts_data = wrapped.get_alerts()

import json
with open('website_data.json', 'w') as f:
    json.dump({
        'history': metrics_history.to_dict('records'),
        'alerts': alerts_data.to_dict('records')
    }, f)

wrapped.close()
print("✓ Exported to website_data.json")
print("✓ Ready to use in website!")

Database created
Generating demo data for website...
Logged 100 predictions
logged metrics (batch: 1)
ALERT: critical - statistical_parity_difference = -0.2877
ALERT: warning - equal_opportunity_difference = -0.0833
ALERT: critical - predictive_parity_difference = -0.1413
ALERT: critical - accuracy_difference = 0.1905
  batch 1 complete
Logged 100 predictions
logged metrics (batch: 2)
ALERT: critical - statistical_parity_difference = -0.2596
ALERT: warning - equal_opportunity_difference = 0.0625
ALERT: warning - predictive_parity_difference = -0.0833
ALERT: warning - accuracy_difference = 0.0737
  batch 2 complete
Logged 100 predictions
logged metrics (batch: 3)
ALERT: critical - statistical_parity_difference = -0.2806
ALERT: critical - equal_opportunity_difference = -0.1583
ALERT: critical - predictive_parity_difference = -0.2525
  batch 3 complete
Logged 100 predictions


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Do

logged metrics (batch: 4)
ALERT: critical - statistical_parity_difference = -0.5542
ALERT: critical - equal_opportunity_difference = -0.1974
ALERT: critical - predictive_parity_difference = 0.1694
ALERT: critical - accuracy_difference = 0.2465
  batch 4 complete
Logged 100 predictions
logged metrics (batch: 5)
ALERT: critical - statistical_parity_difference = -0.3141
ALERT: warning - equal_opportunity_difference = 0.0667
ALERT: critical - predictive_parity_difference = 0.2400
ALERT: critical - accuracy_difference = 0.2057
  batch 5 complete
degraded 99 numeric feaetures by 40%
Logged 100 predictions
logged metrics (batch: 6)
ALERT: critical - statistical_parity_difference = -0.4600
ALERT: critical - equal_opportunity_difference = -0.4444
ALERT: critical - predictive_parity_difference = 0.5143
ALERT: critical - accuracy_difference = 0.3000
  batch 6 complete
degraded 99 numeric feaetures by 40%


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Do

Logged 100 predictions
logged metrics (batch: 7)
ALERT: critical - statistical_parity_difference = -0.2649
ALERT: critical - equal_opportunity_difference = -0.5286
ALERT: critical - predictive_parity_difference = -0.4342
  batch 7 complete
degraded 99 numeric feaetures by 40%
Logged 100 predictions
logged metrics (batch: 8)
ALERT: critical - statistical_parity_difference = -0.1114
ALERT: critical - equal_opportunity_difference = 0.2222
ALERT: warning - predictive_parity_difference = 0.0530
ALERT: critical - accuracy_difference = 0.1269
  batch 8 complete
degraded 99 numeric feaetures by 40%
Logged 100 predictions
logged metrics (batch: 9)
ALERT: critical - statistical_parity_difference = -0.1925
ALERT: critical - predictive_parity_difference = 0.2773
ALERT: critical - accuracy_difference = 0.1673
  batch 9 complete
degraded 99 numeric feaetures by 40%
Logged 100 predictions
logged metrics (batch: 10)
ALERT: critical - statistical_parity_difference = -0.3173
ALERT: critical - accuracy_d

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.0s finished
